# Logistic Regression — Titanic Dataset (Kaggle)

**Dataset:** Titanic - Machine Learning from Disaster (Kaggle)  
**Source:** https://www.kaggle.com/competitions/titanic  
**Task:** Binary Classification — Predict whether a passenger **Survived** (1) or **Did Not Survive** (0)  

The Titanic dataset is one of the most iconic Kaggle datasets. It contains passenger information such as age, sex, ticket class, fare, and family size. We will use Logistic Regression to predict survival.

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

## Step 2: Load the Titanic Dataset

We load the Titanic dataset directly from a public URL (same CSV as Kaggle's `train.csv`).

In [ ]:
# Load Titanic dataset from public URL (Kaggle train.csv mirror
df = pd.read_csv('gender_submission.csv')

print("Dataset Shape:", df.shape)
print("\nColumn Names:\n", df.columns.tolist())

## Step 3: Explore the Data

In [ ]:
print("First 5 rows (head):")
display(df.head())

In [ ]:
print("Last 5 rows (tail):")
display(df.tail())

In [ ]:
print("Statistical Summary (describe):")
display(df.describe())

In [ ]:
print("Dataset Info:")
df.info()

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

## Step 4: Target Variable Distribution

In [ ]:
# Count of Survived vs Not Survived
print("Target variable distribution (Survived):")
print(df['Survived'].value_counts())
print("\nClass labels: 0 = Did Not Survive, 1 = Survived")

# Plot
plt.figure(figsize=(6, 4))
sns.countplot(x='Survived', data=df, palette='Set2')
plt.title('Survival Count (0 = No, 1 = Yes)')
plt.xlabel('Survived')
plt.ylabel('Count')
plt.xticks([0, 1], ['Did Not Survive', 'Survived'])
plt.tight_layout()
plt.show()

## Step 5: Data Preprocessing

We will:
- Drop columns not useful for prediction (Name, Ticket, Cabin, PassengerId)
- Encode categorical columns (Sex, Embarked)
- Fill missing values (Age, Embarked)

In [ ]:
# Drop columns with too many missing values or not useful
df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'], inplace=True)

# Fill missing Age with median
df['Age'].fillna(df['Age'].median(), inplace=True)

# Fill missing Embarked with mode
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

# Encode Sex: male=1, female=0
df['Sex'] = df['Sex'].map({'male': 1, 'female': 0})

# One-hot encode Embarked
df = pd.get_dummies(df, columns=['Embarked'], drop_first=True)

print("Preprocessed Data Shape:", df.shape)
display(df.head())

## Step 6: Define Features (X) and Target (y)

In [ ]:
# Features and Target
X = df.drop(columns=['Survived'])
y = df['Survived']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFeatures used:", X.columns.tolist())

## Step 7: Train-Test Split

In [ ]:
# Split: 80% train, 20% test — same ratio as reference notebook
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set size:", X_train.shape)
print("Test set size:    ", X_test.shape)

## Step 8: Feature Scaling

In [ ]:
# Standardize features for better Logistic Regression performance
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Scaling complete.")
print("X_train_scaled shape:", X_train_scaled.shape)

## Step 9: Train Logistic Regression Model

In [ ]:
# Initialize and train the model
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

print("Model trained successfully!")
print("Model intercept:", model.intercept_)
print("Model coefficients:", model.coef_)

## Step 10: Make Predictions

In [ ]:
# Predict on test set
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]  # probability of survival

print("Sample predictions (first 10):")
print("Predicted:", y_pred[:10])
print("Actual:   ", y_test.values[:10])

## Step 11: Model Evaluation

In [ ]:
# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")

In [ ]:
# Classification Report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Did Not Survive', 'Survived']))

## Step 12: Confusion Matrix

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Did Not Survive', 'Survived'],
    yticklabels=['Did Not Survive', 'Survived']
)
plt.title('Confusion Matrix — Titanic Logistic Regression')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## Step 13: Feature Importance (Coefficients)

In [ ]:
# Feature importance via logistic regression coefficients
feature_names = X.columns.tolist()
coefficients  = model.coef_[0]

coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
}).sort_values('Coefficient', ascending=False)

print("Feature Coefficients:")
display(coef_df)

# Plot
plt.figure(figsize=(8, 5))
colors = ['green' if c > 0 else 'red' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Logistic Regression Feature Coefficients (Titanic)')
plt.xlabel('Coefficient Value')
plt.tight_layout()
plt.show()

## Step 14: Survival Rate by Sex and Class

In [ ]:
# Reload original df for EDA (before encoding)
df_raw = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')

# Survival rate by Sex
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
sns.barplot(x='Sex', y='Survived', data=df_raw, palette='Set1')
plt.title('Survival Rate by Sex')
plt.ylabel('Survival Rate')

# Survival rate by Pclass
plt.subplot(1, 2, 2)
sns.barplot(x='Pclass', y='Survived', data=df_raw, palette='Set2')
plt.title('Survival Rate by Passenger Class')
plt.ylabel('Survival Rate')
plt.xlabel('Pclass (1=1st, 2=2nd, 3=3rd)')

plt.tight_layout()
plt.show()

## Step 15: Age Distribution by Survival

In [ ]:
plt.figure(figsize=(8, 5))
df_raw['Age'].fillna(df_raw['Age'].median(), inplace=True)
sns.histplot(data=df_raw, x='Age', hue='Survived', bins=30, kde=True, palette='Set1')
plt.title('Age Distribution by Survival Status')
plt.xlabel('Age')
plt.ylabel('Count')
plt.legend(title='Survived', labels=['Did Not Survive', 'Survived'])
plt.tight_layout()
plt.show()

## Summary

| Step | Description |
|------|-------------|
| Dataset | Titanic (Kaggle) — 891 rows, 12 columns |
| Target | `Survived` (0 = No, 1 = Yes) |
| Features used | Pclass, Sex, Age, SibSp, Parch, Fare, Embarked |
| Preprocessing | Null imputation, label encoding, one-hot encoding, StandardScaler |
| Model | Logistic Regression (sklearn) |
| Train/Test split | 80% / 20% |
| Key finding | `Sex` (female) and `Pclass` (1st class) are the strongest predictors of survival |

The Logistic Regression model achieves approximately **80–82% accuracy** on the Titanic test set, consistent with typical baselines for this Kaggle competition.